# Retina Severity Classification

**Five-class retinal severity classification using EfficientNet-B0 transfer learning**

This notebook contains the cleaned, reproducible version of the final baseline experiment developed from the original working notebook.

### Dataset
- APTOS: 3,662 images
- EyePACS: 35,108 images
- Combined: 38,770 images
- Classes: 0, 1, 2, 3, 4
- Stratified train/validation split: 85% / 15%

### Final documented run
- Model: ImageNet-pretrained EfficientNet-B0
- Input size: 224 × 224
- Batch size: 32
- Optimizer: AdamW
- Initial learning rate: 3e-4
- Weight decay: 1e-4
- Loss: normalized class-weighted cross-entropy
- Scheduler: ReduceLROnPlateau on validation macro-F1
- Mixed precision: CUDA AMP
- Best validation macro-F1: **0.5411**
- Best checkpoint: **Epoch 8 of the documented run**

> This is a research/educational model and is not a clinically validated diagnostic system.

## 1. Project setup

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Project root is one level above the notebooks directory.
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

APTOS_CSV = RAW_DIR / "train.csv"
APTOS_IMAGE_DIR = RAW_DIR / "train_images"

EYEPACS_CSV = RAW_DIR / "eyepacs" / "trainLabels_cropped.csv"
EYEPACS_IMAGE_DIR = RAW_DIR / "eyepacs" / "resized_train_cropped" / "resized_train_cropped"

MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5
VAL_SIZE = 0.15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

## 2. Load and standardize APTOS and EyePACS

In [2]:
# APTOS
aptos_df = pd.read_csv(APTOS_CSV)
aptos_df = aptos_df[["id_code", "diagnosis"]].copy()
aptos_df["filepath"] = aptos_df["id_code"].map(
    lambda x: APTOS_IMAGE_DIR / f"{x}.png"
)
aptos_df["source"] = "aptos"

# EyePACS
eyepacs_df = pd.read_csv(EYEPACS_CSV)
eyepacs_df = eyepacs_df[["image", "level"]].copy()

eyepacs_df.columns = ["id_code", "diagnosis"]
eyepacs_df["filepath"] = eyepacs_df["id_code"].map(
    lambda x: EYEPACS_IMAGE_DIR / f"{x}.jpeg"
)
eyepacs_df["source"] = "eyepacs"

# Combine
combined_df = pd.concat([aptos_df, eyepacs_df], ignore_index=True)

print("APTOS samples:", len(aptos_df))
print("EyePACS samples:", len(eyepacs_df))
print("Combined samples:", len(combined_df))
print("\nClass distribution:")
print(combined_df["diagnosis"].value_counts().sort_index())

APTOS samples: 3662
EyePACS samples: 35108
Combined samples: 38770

Class distribution:
diagnosis
0    27607
1     2808
2     6287
3     1065
4     1003
Name: count, dtype: int64


## 3. Verify the dataset

In [3]:
# Verify every referenced image exists.
exists = combined_df["filepath"].map(Path.exists)

print("Total images:", len(combined_df))
print("Existing:", exists.sum())
print("Missing:", (~exists).sum())

if not exists.all():
    print("\nExample missing paths:")
    print(combined_df.loc[~exists, "filepath"].head())
    raise FileNotFoundError("One or more image paths are missing.")

# Verify IDs.
duplicate_ids = combined_df["id_code"].duplicated().sum()

print("\nUnique IDs:", combined_df["id_code"].nunique())
print("Duplicate IDs:", duplicate_ids)

# Open a few images to verify readability.
for _, row in combined_df.sample(5, random_state=SEED).iterrows():
    with Image.open(row["filepath"]) as img:
        print(
            f"{row['id_code']} | diagnosis={row['diagnosis']} | "
            f"size={img.size} | mode={img.mode}"
        )

Total images: 38770
Existing: 38770
Missing: 0

Unique IDs: 38770
Duplicate IDs: 0
25852_right | diagnosis=0 | size=(1024, 1024) | mode=RGB
1698_left | diagnosis=0 | size=(1024, 826) | mode=RGB
43912_left | diagnosis=0 | size=(1024, 827) | mode=RGB
fdbc252813b1 | diagnosis=0 | size=(819, 614) | mode=RGB
19983_left | diagnosis=0 | size=(1024, 819) | mode=RGB


## 4. Stratified train/validation split

In [4]:
train_df, val_df = train_test_split(
    combined_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=combined_df["diagnosis"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

print("\nTraining distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["diagnosis"].value_counts().sort_index())

Training samples: 32954
Validation samples: 5816

Training distribution:
diagnosis
0    23466
1     2387
2     5344
3      905
4      852
Name: count, dtype: int64

Validation distribution:
diagnosis
0    4141
1     421
2     943
3     160
4     151
Name: count, dtype: int64


## 5. Dataset class and image transforms

In [5]:
class RetinaDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        with Image.open(row["filepath"]) as image:
            image = image.convert("RGB")

        label = int(row["diagnosis"])

        if self.transform:
            image = self.transform(image)

        return image, label


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = RetinaDataset(train_df, transform=train_transform)
val_dataset = RetinaDataset(val_df, transform=val_transform)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 32954
Validation dataset: 5816


## 6. DataLoaders

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)

Train batches: 1030
Validation batches: 182
Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])


## 7. Class weights and loss

In [7]:
class_counts = train_df["diagnosis"].value_counts().sort_index()

# Inverse-frequency weights.
raw_weights = len(train_df) / (NUM_CLASSES * class_counts.values)

# Normalize weights so their mean is 1.
class_weights = torch.tensor(raw_weights, dtype=torch.float32)
class_weights = class_weights / class_weights.mean()

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

print("Training class counts:")
print(class_counts)
print("\nNormalized class weights:")
print(class_weights)

Training class counts:
diagnosis
0    23466
1     2387
2     5344
3      905
4      852
Name: count, dtype: int64

Normalized class weights:
tensor([0.0728, 0.7156, 0.3196, 1.8873, 2.0047])


## 8. EfficientNet-B0 model

In [8]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, NUM_CLASSES)

model = model.to(device)

print(model.classifier)

# Forward-pass sanity check.
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Input shape:", images.shape)
print("Output shape:", sample_outputs.shape)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=5, bias=True)
)
Input shape: torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 5])


## 9. Optimizer, scheduler, and mixed precision

In [9]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print("Optimizer:", optimizer)
print("Scheduler:", scheduler)
print("Mixed precision enabled:", torch.cuda.is_available())

Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0.0001
)
Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x000002A05C41FB90>
Mixed precision enabled: True


## 10. Training and validation functions

In [10]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Validation", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1

## 11. Train the baseline

In [11]:
NUM_EPOCHS = 8

history = []
best_val_f1 = -np.inf
best_model_state = None

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler
    )

    val_loss, val_acc, val_f1 = validate(
        model, val_loader, criterion, device
    )

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "lr": current_lr
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.4f} | "
        f"Train macro-F1: {train_f1:.4f}"
    )
    print(
        f"Val loss: {val_loss:.4f} | "
        f"Val acc: {val_acc:.4f} | "
        f"Val macro-F1: {val_f1:.4f}"
    )
    print(f"Learning rate: {current_lr:.6f}")


Epoch 1/8


Training:   0%|          | 0/1030 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 12. Training curves

In [ ]:
history_df = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history_df["epoch"], history_df["train_f1"], marker="o", label="Train macro-F1")
ax.plot(history_df["epoch"], history_df["val_f1"], marker="o", label="Validation macro-F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Macro-F1")
ax.set_title("Macro-F1 during training")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_macro_f1.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
ax.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Loss during training")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Restore the best checkpoint

In [ ]:
model.load_state_dict(best_model_state)
model = model.to(device)

print(f"Best validation Macro-F1: {best_val_f1:.4f}")

# Save the best checkpoint for reproducibility.
checkpoint_path = MODEL_DIR / "retina_severity_efficientnet_b0_best.pth"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_val_macro_f1": float(best_val_f1),
        "img_size": IMG_SIZE,
        "num_classes": NUM_CLASSES,
        "class_weights": class_weights.cpu()
    },
    checkpoint_path
)

print("Checkpoint saved to:", checkpoint_path)

## 14. Final validation evaluation

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

report = classification_report(
    all_labels,
    all_preds,
    labels=[0, 1, 2, 3, 4],
    target_names=[
        "Severity 0",
        "Severity 1",
        "Severity 2",
        "Severity 3",
        "Severity 4"
    ],
    digits=4,
    zero_division=0
)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4])

print(report)
print("Confusion Matrix:")
print(cm)

with open(OUTPUT_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(np.array2string(cm))

np.savetxt(OUTPUT_DIR / "confusion_matrix.csv", cm, fmt="%d", delimiter=",")

## 15. Documented result from the completed development run

The completed run used during development produced:

- **Best validation macro-F1: 0.5411**
- Validation accuracy at the best checkpoint: **0.6730**
- Best checkpoint: **Epoch 8**

Per-class F1 from that checkpoint:

| Severity | Precision | Recall | F1 |
|---|---:|---:|---:|
| 0 | 0.9089 | 0.7254 | 0.8069 |
| 1 | 0.1501 | 0.3919 | 0.2171 |
| 2 | 0.5433 | 0.5790 | 0.5606 |
| 3 | 0.4235 | 0.6750 | 0.5205 |
| 4 | 0.5987 | 0.6026 | 0.6007 |
| **Macro avg** | **0.5249** | **0.5948** | **0.5411** |

The model performs substantially better on Severity 0, 2, 3, and 4 than on Severity 1. This limitation should remain visible in the project documentation rather than being hidden by overall accuracy.

## 16. Limitations

- The validation split is image-level and stratified; the available dataframe contains unique `id_code` values, but unique image IDs do not by themselves prove patient-level independence.
- The combined dataset is strongly class-imbalanced.
- Images are resized to 224 × 224, which may discard small retinal features.
- The model has not been clinically validated.
- The reported validation results are from a single development split.
- Dataset licensing and redistribution terms must be checked before redistributing any image data.
- This project is intended for research/educational purposes and should not be used for medical diagnosis.

## 17. Conclusion

This baseline demonstrates an end-to-end five-class retinal severity classification pipeline using EfficientNet-B0 transfer learning, data augmentation, class-weighted cross-entropy, mixed-precision training, and macro-F1-based checkpoint selection.

The documented development run achieved a best validation macro-F1 of **0.5411**. Future experiments can investigate alternative imbalance strategies, loss functions, image resolution, and ordinal-aware methods.